In [4]:
## installation part

In [1]:
!pip install transformers

Defaulting to user installation because normal site-packages is not writeable

[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python3 -m pip install --upgrade pip


In [1]:
import torch
import torch.nn as nn
from transformers import BertModel

### Read Dataset

In [2]:
import pandas as pd

df = pd.read_csv("/home/firas/Sentiment analysis bert/data/IMDB Dataset.csv")

In [3]:
df.head()

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


##### Load Pretrained BERT

In [4]:
from transformers import BertModel

bert = BertModel.from_pretrained(
    "bert-base-uncased"
)

#### Convert Labels

In [5]:
df["label"] = df["sentiment"].map({
    "negative": 0,
    "positive": 1
})

#### Train/Test Split

In [6]:
from sklearn.model_selection import train_test_split

train_texts, test_texts, train_labels, test_labels = train_test_split(
    df["review"],
    df["label"],
    test_size=0.2,
    random_state=42
)

## Phase 2 — Tokenization

### Load Tokenizer

In [7]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(
    "bert-base-uncased"
)

### Tokenize Training Reviews

In [8]:
train_encodings = tokenizer(
    train_texts.tolist(),
    truncation=True,
    padding=True,
    max_length=128
)

test_encodings = tokenizer(
    test_texts.tolist(),
    truncation=True,
    padding=True,
    max_length=128
)

### Phase 3 — Dataset Class

In [9]:
import torch
from torch.utils.data import Dataset

class IMDBDataset(Dataset):

    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):

        item = {
            key: torch.tensor(val[idx])
            for key, val in self.encodings.items()
        }

        item["labels"] = torch.tensor(
            self.labels.iloc[idx]
        )

        return item

    def __len__(self):
        return len(self.labels)

#### Create Dataset

In [10]:
train_dataset = IMDBDataset(
    train_encodings,
    train_labels
)

test_dataset = IMDBDataset(
    test_encodings,
    test_labels
)

In [11]:
sample = train_dataset[0]

print(sample.keys())

dict_keys(['input_ids', 'token_type_ids', 'attention_mask', 'labels'])


## Create DataLoader
Because BERT doesn't train one review at a time.

It trains in batches.

Review 1
Review 2
Review 3
Review 4
↓
One batch

In [12]:
from torch.utils.data import DataLoader

train_loader = DataLoader(
    train_dataset,
    batch_size=8,
    shuffle=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=8
)

### Phase 4 — Model

In [13]:
device = torch.device("cuda")



#### Create Our  Model

In [14]:
class SentimentClassifier(nn.Module):

    def __init__(self):

        super().__init__()

        self.bert = BertModel.from_pretrained(
            "bert-base-uncased"
        )

        self.dropout = nn.Dropout(0.3)

        self.classifier = nn.Linear(
            768,
            2
        )

    def forward(
        self,
        input_ids,
        attention_mask
    ):

        outputs = self.bert(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        pooled_output = outputs.pooler_output

        pooled_output = self.dropout(
            pooled_output
        )

        logits = self.classifier(
            pooled_output
        )

        return logits

In [15]:
model = SentimentClassifier()

In [16]:
model = model.to(device)

In [17]:
criterion = nn.CrossEntropyLoss()

In [18]:
from torch.optim import AdamW

optimizer = AdamW(
    model.parameters(),
    lr=2e-5
)

In [19]:
sample = next(iter(train_loader))

for key, value in sample.items():
    print(key, value.shape)

input_ids torch.Size([8, 128])
token_type_ids torch.Size([8, 128])
attention_mask torch.Size([8, 128])
labels torch.Size([8])


In [20]:
sample = train_dataset[0]

input_ids = sample["input_ids"].unsqueeze(0).to(device)
attention_mask = sample["attention_mask"].unsqueeze(0).to(device)

logits = model(
    input_ids=input_ids,
    attention_mask=attention_mask
)

print(logits)
print(logits.shape)

tensor([[ 0.2926, -0.3692]], device='cuda:0', grad_fn=<AddmmBackward0>)
torch.Size([1, 2])


In [21]:
print(input_ids.device)
print(next(model.parameters()).device)

cuda:0
cuda:0


In [22]:
sample = next(iter(train_loader))

input_ids = sample["input_ids"].to(device)
attention_mask = sample["attention_mask"].to(device)

logits = model(
    input_ids=input_ids,
    attention_mask=attention_mask
)

print(logits.shape)

torch.Size([8, 2])


In [23]:
batch = next(iter(train_loader))

input_ids = batch["input_ids"].to(device)
attention_mask = batch["attention_mask"].to(device)
labels = batch["labels"].to(device)

In [24]:
print("input_ids:", input_ids.device)
print("attention_mask:", attention_mask.device)
print("model:", next(model.parameters()).device)

input_ids: cuda:0
attention_mask: cuda:0
model: cuda:0


In [25]:
logits = model(
    input_ids=input_ids,
    attention_mask=attention_mask
)

print(logits.shape)

torch.Size([8, 2])


In [26]:
loss = criterion(
    logits,
    labels
)

print(loss)

tensor(0.7309, device='cuda:0', grad_fn=<NllLossBackward0>)


In [27]:
loss.backward() ##Compute Gradients

In [28]:
optimizer.step()

In [29]:
optimizer.zero_grad()

In [30]:
print(loss.item())

0.7308852076530457


In [31]:
## train model on multiple batch 

In [32]:
print(torch.cuda.get_device_name(0))

NVIDIA A40-24Q


In [33]:
print(next(model.parameters()).device)

print(torch.cuda.memory_allocated()/1024**3)

cuda:0
1.246908187866211


In [35]:
print(input_ids.device)
print(attention_mask.device)
print(labels.device)

cuda:0
cuda:0
cuda:0


In [37]:
print(next(model.parameters()).device)

cuda:0


In [38]:
logits = model(
    input_ids=input_ids,
    attention_mask=attention_mask
)

print(logits.shape)

torch.Size([8, 2])


In [40]:
from tqdm import tqdm

model.train()

total_loss = 0

for batch in tqdm(train_loader):

    input_ids = batch["input_ids"].to(device)
    attention_mask = batch["attention_mask"].to(device)
    labels = batch["labels"].to(device)

    logits = model(
        input_ids=input_ids,
        attention_mask=attention_mask
    )

    loss = criterion(
        logits,
        labels
    )

    total_loss += loss.item()

    optimizer.zero_grad()

    loss.backward()

    optimizer.step()

print(
    "Average Loss:",
    total_loss / len(train_loader)
)

100%|██████████| 5000/5000 [04:39<00:00, 17.87it/s]

Average Loss: 0.26104662145264446


In [41]:
import torch

model.eval()

correct = 0
total = 0

with torch.no_grad():

    for batch in test_loader:

        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        logits = model(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        predictions = torch.argmax(
            logits,
            dim=1
        )

        correct += (
            predictions == labels
        ).sum().item()

        total += labels.size(0)

accuracy = correct / total

print(
    f"Accuracy: {accuracy:.4f}"
)

Accuracy: 0.8920
